#### Conversational AI aka ChatBot

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)

GEMINI_BASE_URL = os.getenv('GEMINI_BASE_URL')
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

if GEMINI_API_KEY:
    print(f'Gemini API Key found and starts with {GEMINI_API_KEY[0:3]}')
else:
    print('Gemini API Key not found')

gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=GEMINI_API_KEY)

Gemini API Key found and starts with AQ.


In [2]:
SYSTEM_MESSAGE = "You are a helpful assistant!"

In [3]:
def chatbot(message, history):
    try:
        history = [{'role':h['role'], 'content':h['content']} for h in history]
        messages = [{'role':'system', 'content':SYSTEM_MESSAGE}] + history + [{'role':'user', 'content':message}]
        response = gemini.chat.completions.create(
            model = 'gemini-3.5-flash',
            messages = messages
        )
        return response.choices[0].message.content
    except Exception as e:
        return f'Exception: {e}'


In [4]:
gr.ChatInterface(fn=chatbot).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [5]:
''' 
Note:

The reason for 
history = [{'role':h['role'], 'content':h['content']} for h in history]

is because, gemini model does consist of metadata and if we fed back the history along with metadata,
gemini models won't perform as per the expectations.

So, it is always better to select the 'role' and 'content' attributes from the history and feed back them into the message.

This is same for GROQ models as well

'''

" \nNote:\n\nThe reason for \nhistory = [{'role':h['role'], 'content':h['content']} for h in history]\n\nis because, gemini model does consist of metadata and if we fed back the history along with metadata,\ngemini models won't perform as per the expectations.\n\nSo, it is always better to select the 'role' and 'content' attributes from the history and feed back them into the message.\n\nThis is same for GROQ models as well\n\n"

#### One Shot Prompting

Using a system message to add context and using one shot prompting to give an example

In [6]:
SYSTEM_MESSAGE = "You are a helpful assistant in a clothes store. You should try to gently encourage \
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. \
For example, if the customer says 'I'm looking to buy a hat', \
you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'\
Encourage the customer to buy hats if they are unsure what to get."

In [7]:
gr.ChatInterface(fn=chatbot).launch()

* Running on local URL:  http://127.0.0.1:7888
* To create a public link, set `share=True` in `launch()`.


In [8]:
# Additional context can be appended to the system message

SYSTEM_MESSAGE += "\nIf the customer asks for shoes, you should respond that shoes are not on sale today, \
but remind the customer to look at hats!"

In [9]:
gr.ChatInterface(fn=chatbot).launch()

* Running on local URL:  http://127.0.0.1:7889
* To create a public link, set `share=True` in `launch()`.


In [10]:
def chatbot(message, history):
    history = [{'role': h['role'], 'content':h['content']} for h in history]
    relevant_system_message = SYSTEM_MESSAGE
    if 'belt' in message.lower():
        relevant_system_message += "\n The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."
    messages = [{'role':'system', 'content':relevant_system_message}] + history + [{'role':'user', 'content':message}]

    try:
        response = gemini.chat.completions.create(
            model = 'gemini-3.5-flash-lite',
            messages = messages
        )
        return response.choices[0].message.content
    except Exception as e:
        return f'Exception: {e}'

    ''' 
    This technique of using conditional statements for system prompts is very helpful \
    especially when the context is too big.
    In this approach, context becomes big only when a certain condition is met i.e., \
    a word name 'belt' appears in the input message. \
    Otherwise, system prompt remains small which saves the tokens cost
    '''

In [11]:
gr.ChatInterface(fn=chatbot).launch()

* Running on local URL:  http://127.0.0.1:7890
* To create a public link, set `share=True` in `launch()`.


#### Streaming ChatBot

In [12]:
def chatbot(message, history):
    history = [{'role': h['role'], 'content':h['content']} for h in history]
    relevant_system_message = SYSTEM_MESSAGE
    if 'belt' in message.lower():
        relevant_system_message += "\n The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."
    messages = [{'role':'system', 'content':relevant_system_message}] + history + [{'role':'user', 'content':message}]

    try:
        stream = gemini.chat.completions.create(
            model = 'gemini-3.5-flash-lite',
            messages = messages,
            stream = True
        )
        response = ""
        for chunk in stream:
            response += chunk.choices[0].delta.content or ""
            yield response
    except Exception as e:
        yield f'Exception: {e}'

In [13]:
gr.ChatInterface(fn=chatbot).launch()

* Running on local URL:  http://127.0.0.1:7891
* To create a public link, set `share=True` in `launch()`.
